# Matched-Protocol Multi-Seed Replication

Matched-protocol training and evaluation for seeds 13, 42, and 100.

Set the private project root before execution:

```python
import os
os.environ['KLJPAUDIT_ROOT'] = '/path/to/K-LJPAUDIT'
```

Do not publish raw judicial facts, transformed text, per-case predictions, case identifiers, review files, or model checkpoints.

## Execution block 1

In [ ]:
import os
# ============================================================
# K-LJPAUDIT
# MATCHED-PROTOCOL SEED-42 RETRAINING + MULTI-SEED AGGREGATION
# ============================================================

# 필요하면 최초 1회만 설치
# !pip -q install -U datasets transformers scikit-learn pyarrow

from google.colab import drive
drive.mount("/content/drive")

import gc
import json
import math
import random
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(os.environ.get("KLJPAUDIT_ROOT", "."))

EXPERIMENT_NAME = "baseline_multiseed_replication_v1"
RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "audit_experiments"
    / EXPERIMENT_NAME
)

CHECKPOINT_ROOT = (
    PROJECT_ROOT
    / "checkpoints"
    / "klue_roberta_base_multiseed"
)

SEED42_CHECKPOINT_DIR = CHECKPOINT_ROOT / "seed_42"
PREDICTION_DIR = RESULT_ROOT / "predictions"
METRICS_DIR = RESULT_ROOT / "metrics"

for path in [
    RESULT_ROOT,
    CHECKPOINT_ROOT,
    PREDICTION_DIR,
    METRICS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "klue/roberta-base"
DATASET_NAME = "lbox/lbox_open"
DATASET_CONFIG = "ljp_criminal"

TRAIN_SEED = 42

TARGETS = [
    "fine_lv",
    "imprisonment_with_labor_lv",
    "imprisonment_without_labor_lv",
]

NUM_LABELS = {
    "fine_lv": 5,
    "imprisonment_with_labor_lv": 6,
    "imprisonment_without_labor_lv": 5,
}

MAX_LENGTH = 512
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
DROPOUT = 0.10
MAX_GRAD_NORM = 1.0
NUM_WORKERS = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

print("=" * 72)
print("MATCHED-PROTOCOL SEED-42 RETRAINING")
print("=" * 72)
print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB",
    )

print("PyTorch:", torch.__version__)
print("Result root:", RESULT_ROOT)
print("Checkpoint directory:", SEED42_CHECKPOINT_DIR)


# ------------------------------------------------------------
# 2. Reproducibility
# ------------------------------------------------------------

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # 재현성을 우선
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


set_seed(TRAIN_SEED)


# ------------------------------------------------------------
# 3. Backup any existing seed-42 matched checkpoint
# ------------------------------------------------------------

# 잘못 생성되거나 중단된 seed_42 폴더가 있으면 백업하고 새로 학습
if SEED42_CHECKPOINT_DIR.exists():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_dir = CHECKPOINT_ROOT / f"seed_42_backup_{timestamp}"

    print("\nExisting seed-42 directory detected.")
    print("Backing up:")
    print(" ", SEED42_CHECKPOINT_DIR)
    print(" ->", backup_dir)

    shutil.move(str(SEED42_CHECKPOINT_DIR), str(backup_dir))

SEED42_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 4. Load LBox criminal LJP dataset
# ------------------------------------------------------------

print("\nLoading dataset...")

raw_dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG,
)

required_splits = {"train", "validation", "test"}

# 일부 datasets 버전에서는 valid 대신 validation으로 로드될 수 있음
if "valid" in raw_dataset and "validation" not in raw_dataset:
    raw_dataset["validation"] = raw_dataset["valid"]

missing_splits = required_splits - set(raw_dataset.keys())
assert not missing_splits, (
    f"Missing dataset splits: {sorted(missing_splits)}; "
    f"available={list(raw_dataset.keys())}"
)

train_raw = raw_dataset["train"]
valid_raw = raw_dataset["validation"]
test_raw = raw_dataset["test"]

assert len(train_raw) == 8400, f"Expected 8,400 train cases, got {len(train_raw)}"
assert len(valid_raw) == 1050, f"Expected 1,050 valid cases, got {len(valid_raw)}"
assert len(test_raw) == 1050, f"Expected 1,050 test cases, got {len(test_raw)}"

print("Train:", len(train_raw))
print("Valid:", len(valid_raw))
print("Test :", len(test_raw))


def extract_labels(example):
    label_obj = example["label"]

    return {
        target: int(label_obj[target])
        for target in TARGETS
    }


for split_name, split_data in [
    ("train", train_raw),
    ("validation", valid_raw),
    ("test", test_raw),
]:
    assert "facts" in split_data.column_names
    assert "label" in split_data.column_names

    for i in range(min(100, len(split_data))):
        labels = extract_labels(split_data[i])

        for target in TARGETS:
            value = labels[target]
            assert 0 <= value < NUM_LABELS[target], (
                f"{split_name}[{i}] invalid {target}: {value}"
            )

print("Dataset validation: PASSED")


# ------------------------------------------------------------
# 5. Tokenizer and PyTorch dataset
# ------------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
)


class CriminalLJPDataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_length):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data[index]

        text = str(row["facts"])
        labels = extract_labels(row)

        encoded = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding=False,
            return_attention_mask=True,
        )

        item = {
            "case_uid": str(row["id"]),
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
        }

        for target in TARGETS:
            item[target] = labels[target]

        return item


def collate_batch(batch):
    token_features = [
        {
            "input_ids": row["input_ids"],
            "attention_mask": row["attention_mask"],
        }
        for row in batch
    ]

    padded = tokenizer.pad(
        token_features,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    output = {
        "case_uid": [row["case_uid"] for row in batch],
        "input_ids": padded["input_ids"],
        "attention_mask": padded["attention_mask"],
    }

    for target in TARGETS:
        output[target] = torch.tensor(
            [row[target] for row in batch],
            dtype=torch.long,
        )

    return output


train_dataset = CriminalLJPDataset(train_raw, tokenizer, MAX_LENGTH)
valid_dataset = CriminalLJPDataset(valid_raw, tokenizer, MAX_LENGTH)
test_dataset = CriminalLJPDataset(test_raw, tokenizer, MAX_LENGTH)


def make_train_loader(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    return DataLoader(
        train_dataset,
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=NUM_WORKERS,
        pin_memory=USE_AMP,
        collate_fn=collate_batch,
    )


valid_loader = DataLoader(
    valid_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    collate_fn=collate_batch,
)


# ------------------------------------------------------------
# 6. Sqrt-inverse class weights
# ------------------------------------------------------------

train_label_arrays = {
    target: np.array(
        [int(row["label"][target]) for row in train_raw],
        dtype=np.int64,
    )
    for target in TARGETS
}


def make_sqrt_inverse_weights(labels, num_labels):
    counts = np.bincount(labels, minlength=num_labels).astype(np.float64)

    assert np.all(counts > 0), (
        f"At least one class has zero training support: {counts.tolist()}"
    )

    weights = 1.0 / np.sqrt(counts)

    # 평균을 1로 정규화
    weights = weights / weights.mean()

    return counts, torch.tensor(weights, dtype=torch.float32)


class_weights = {}

print("\nClass distributions and weights:")

for target in TARGETS:
    counts, weights = make_sqrt_inverse_weights(
        train_label_arrays[target],
        NUM_LABELS[target],
    )

    class_weights[target] = weights.to(DEVICE)

    print(f"\n[{target}]")
    print(" Counts :", counts.astype(int).tolist())
    print(" Weights:", np.round(weights.numpy(), 6).tolist())


# ------------------------------------------------------------
# 7. Multi-task classifier
# ------------------------------------------------------------

class MultiTaskClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.heads = nn.ModuleDict({
            target: nn.Linear(hidden_size, n_labels)
            for target, n_labels in num_labels.items()
        })

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # 기존 실험과 같은 CLS representation
        cls_representation = outputs.last_hidden_state[:, 0, :]
        cls_representation = self.dropout(cls_representation)

        logits = {
            target: head(cls_representation)
            for target, head in self.heads.items()
        }

        return logits


model = MultiTaskClassifier(
    model_name=MODEL_NAME,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

loss_functions = {
    target: nn.CrossEntropyLoss(weight=class_weights[target])
    for target in TARGETS
}


# ------------------------------------------------------------
# 8. Evaluation
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    case_uids = []
    true_values = {target: [] for target in TARGETS}
    pred_values = {target: [] for target in TARGETS}

    for batch in loader:
        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True,
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True,
        )

        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        case_uids.extend(batch["case_uid"])

        for target in TARGETS:
            y_true = batch[target].cpu().numpy()
            y_pred = logits[target].argmax(dim=-1).cpu().numpy()

            true_values[target].extend(y_true.tolist())
            pred_values[target].extend(y_pred.tolist())

    metrics = {}
    macro_f1_values = []

    for target in TARGETS:
        labels = list(range(NUM_LABELS[target]))

        accuracy = accuracy_score(
            true_values[target],
            pred_values[target],
        )

        macro_f1 = f1_score(
            true_values[target],
            pred_values[target],
            labels=labels,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            true_values[target],
            pred_values[target],
            labels=labels,
            average="weighted",
            zero_division=0,
        )

        metrics[target] = {
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
        }

        macro_f1_values.append(macro_f1)

    mean_macro_f1 = float(np.mean(macro_f1_values))

    true_matrix = np.column_stack([
        np.asarray(true_values[target])
        for target in TARGETS
    ])

    pred_matrix = np.column_stack([
        np.asarray(pred_values[target])
        for target in TARGETS
    ])

    joint_exact_match = float(
        np.mean(np.all(true_matrix == pred_matrix, axis=1))
    )

    metrics["aggregate"] = {
        "mean_macro_f1": mean_macro_f1,
        "joint_exact_match": joint_exact_match,
    }

    prediction_df = pd.DataFrame({
        "case_uid": case_uids,
    })

    for target in TARGETS:
        prediction_df[f"{target}_true"] = true_values[target]
        prediction_df[f"pred_{target}"] = pred_values[target]

    return metrics, prediction_df


# ------------------------------------------------------------
# 9. Train seed 42
# ------------------------------------------------------------

train_loader = make_train_loader(TRAIN_SEED)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

updates_per_epoch = math.ceil(
    len(train_loader) / GRAD_ACCUM_STEPS
)

total_training_steps = updates_per_epoch * NUM_EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP,
)

print("\nTraining configuration:")
print(" Seed:", TRAIN_SEED)
print(" Epochs:", NUM_EPOCHS)
print(" Train batch:", TRAIN_BATCH_SIZE)
print(" Gradient accumulation:", GRAD_ACCUM_STEPS)
print(" Effective batch:", TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS)
print(" Updates/epoch:", updates_per_epoch)
print(" Total updates:", total_training_steps)
print(" Warmup updates:", warmup_steps)
print(" FP16:", USE_AMP)

best_valid_mean_macro_f1 = -np.inf
best_epoch = None
history = []

checkpoint_path = SEED42_CHECKPOINT_DIR / "best_model.pt"
metadata_path = SEED42_CHECKPOINT_DIR / "training_metadata.json"

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    optimizer_updates = 0

    for step, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(
            DEVICE,
            non_blocking=True,
        )
        attention_mask = batch["attention_mask"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = {
            target: batch[target].to(
                DEVICE,
                non_blocking=True,
            )
            for target in TARGETS
        }

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP,
        ):
            logits = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

            target_losses = [
                loss_functions[target](
                    logits[target],
                    labels[target],
                )
                for target in TARGETS
            ]

            # 세 target 손실의 평균
            loss = torch.stack(target_losses).mean()
            scaled_loss = loss / GRAD_ACCUM_STEPS

        scaler.scale(scaled_loss).backward()
        running_loss += float(loss.detach().cpu())

        should_update = (
            step % GRAD_ACCUM_STEPS == 0
            or step == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                MAX_GRAD_NORM,
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            optimizer_updates += 1

        if step % 200 == 0 or step == len(train_loader):
            print(
                f"Epoch {epoch}/{NUM_EPOCHS} | "
                f"batch {step}/{len(train_loader)} | "
                f"loss {running_loss / step:.6f}"
            )

    valid_metrics, _ = evaluate_model(model, valid_loader)

    valid_mean_f1 = valid_metrics["aggregate"]["mean_macro_f1"]
    valid_joint_em = valid_metrics["aggregate"]["joint_exact_match"]

    epoch_record = {
        "epoch": epoch,
        "train_loss": running_loss / len(train_loader),
        "optimizer_updates": optimizer_updates,
        "valid_mean_macro_f1": valid_mean_f1,
        "valid_joint_exact_match": valid_joint_em,
        "valid_target_metrics": {
            target: valid_metrics[target]
            for target in TARGETS
        },
    }

    history.append(epoch_record)

    print(
        f"\nEpoch {epoch} validation | "
        f"mean Macro-F1={valid_mean_f1:.6f} | "
        f"joint EM={valid_joint_em:.6f}"
    )

    if valid_mean_f1 > best_valid_mean_macro_f1:
        best_valid_mean_macro_f1 = valid_mean_f1
        best_epoch = epoch

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "seed": TRAIN_SEED,
                "epoch": epoch,
                "model_name": MODEL_NAME,
                "num_labels": NUM_LABELS,
                "max_length": MAX_LENGTH,
                "valid_mean_macro_f1": valid_mean_f1,
                "valid_joint_exact_match": valid_joint_em,
                "training_protocol": {
                    "epochs": NUM_EPOCHS,
                    "learning_rate": LEARNING_RATE,
                    "train_batch_size": TRAIN_BATCH_SIZE,
                    "eval_batch_size": EVAL_BATCH_SIZE,
                    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,
                    "weight_decay": WEIGHT_DECAY,
                    "warmup_ratio": WARMUP_RATIO,
                    "dropout": DROPOUT,
                    "class_weighting": "sqrt_inverse_mean_normalized",
                },
            },
            checkpoint_path,
        )

        print("Best checkpoint updated:", checkpoint_path)


# ------------------------------------------------------------
# 10. Reload best seed-42 checkpoint and test
# ------------------------------------------------------------

assert checkpoint_path.exists(), (
    f"Best checkpoint was not saved: {checkpoint_path}"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=DEVICE,
    weights_only=False,
)

assert checkpoint["seed"] == 42
assert checkpoint["model_name"] == MODEL_NAME

load_result = model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True,
)

print("\nCheckpoint loading:")
print(" Missing keys:", load_result.missing_keys)
print(" Unexpected keys:", load_result.unexpected_keys)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

test_metrics, seed42_predictions = evaluate_model(
    model,
    test_loader,
)

print("\n" + "=" * 72)
print("MATCHED-PROTOCOL SEED-42 TEST RESULTS")
print("=" * 72)

for target in TARGETS:
    print(
        f"{target:36s} "
        f"Acc={test_metrics[target]['accuracy']:.6f} | "
        f"Macro-F1={test_metrics[target]['macro_f1']:.6f} | "
        f"Weighted-F1={test_metrics[target]['weighted_f1']:.6f}"
    )

print(
    "\nMean Macro-F1:",
    f"{test_metrics['aggregate']['mean_macro_f1']:.6f}",
)
print(
    "Joint Exact Match:",
    f"{test_metrics['aggregate']['joint_exact_match']:.6f}",
)
print("Best validation epoch:", best_epoch)

assert len(seed42_predictions) == 1050
assert seed42_predictions["case_uid"].notna().all()
assert seed42_predictions["case_uid"].nunique() == 1050

seed42_prediction_path = (
    PREDICTION_DIR
    / "seed_42_predictions_private.parquet"
)

seed42_predictions.to_parquet(
    seed42_prediction_path,
    index=False,
)

seed42_metrics_path = (
    METRICS_DIR
    / "seed_42_matched_protocol_metrics.json"
)

training_metadata = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "completed",
    "seed": 42,
    "training_mode": "matched_protocol_retraining",
    "legacy_checkpoint_used": False,
    "model_name": MODEL_NAME,
    "dataset": f"{DATASET_NAME}:{DATASET_CONFIG}",
    "split_sizes": {
        "train": len(train_raw),
        "validation": len(valid_raw),
        "test": len(test_raw),
    },
    "best_epoch": best_epoch,
    "best_valid_mean_macro_f1": best_valid_mean_macro_f1,
    "test_metrics": test_metrics,
    "training_history": history,
    "checkpoint_path": str(checkpoint_path),
    "private_prediction_path": str(seed42_prediction_path),
}

with open(
    metadata_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        training_metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )

with open(
    seed42_metrics_path,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        training_metadata,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\nSaved seed-42 checkpoint:", checkpoint_path)
print("Saved private predictions:", seed42_prediction_path)
print("Saved metadata:", metadata_path)


# ------------------------------------------------------------
# 11. Locate seed 13 and 100 prediction files
# ------------------------------------------------------------

def locate_prediction_file(seed):
    preferred_candidates = [
        PREDICTION_DIR / f"seed_{seed}_predictions_private.parquet",
        PREDICTION_DIR / f"seed_{seed}_predictions.parquet",
        RESULT_ROOT / f"seed_{seed}_predictions_private.parquet",
        RESULT_ROOT / f"seed_{seed}_predictions.parquet",
    ]

    for candidate in preferred_candidates:
        if candidate.exists():
            return candidate

    recursive_candidates = sorted(
        RESULT_ROOT.rglob(f"*seed*{seed}*.parquet")
    )

    valid_candidates = [
        path
        for path in recursive_candidates
        if "prediction" in path.name.lower()
    ]

    if len(valid_candidates) == 1:
        return valid_candidates[0]

    if len(valid_candidates) > 1:
        print(f"\nSeed {seed} prediction candidates:")

        for path in valid_candidates:
            print(" ", path)

        # private prediction 파일 우선
        private_candidates = [
            path
            for path in valid_candidates
            if "private" in path.name.lower()
        ]

        if len(private_candidates) == 1:
            return private_candidates[0]

        raise RuntimeError(
            f"Multiple prediction files found for seed {seed}. "
            "Please retain only the correct file or set the path manually."
        )

    raise FileNotFoundError(
        f"No prediction parquet found for seed {seed} under {RESULT_ROOT}"
    )


prediction_paths = {
    13: locate_prediction_file(13),
    42: seed42_prediction_path,
    100: locate_prediction_file(100),
}

print("\nPrediction files used for final aggregation:")

for seed, path in prediction_paths.items():
    print(f" Seed {seed}: {path}")


# ------------------------------------------------------------
# 12. Normalize existing prediction schemas
# ------------------------------------------------------------

def normalize_prediction_dataframe(path, seed):
    df = pd.read_parquet(path).copy()

    id_candidates = [
        "case_uid",
        "id",
        "case_id",
        "example_id",
    ]

    id_col = next(
        (column for column in id_candidates if column in df.columns),
        None,
    )

    assert id_col is not None, (
        f"Seed {seed}: no ID column found; columns={list(df.columns)}"
    )

    if id_col != "case_uid":
        df = df.rename(columns={id_col: "case_uid"})

    df["case_uid"] = df["case_uid"].astype(str)

    output = pd.DataFrame({
        "case_uid": df["case_uid"],
    })

    for target in TARGETS:
        true_candidates = [
            f"{target}_true",
            f"true_{target}",
            target,
            f"{target}_label",
            f"label_{target}",
        ]

        pred_candidates = [
            f"pred_{target}",
            f"{target}_pred",
            f"prediction_{target}",
            f"{target}_prediction",
            f"pred_original_{target}",
        ]

        true_col = next(
            (column for column in true_candidates if column in df.columns),
            None,
        )

        pred_col = next(
            (column for column in pred_candidates if column in df.columns),
            None,
        )

        assert true_col is not None, (
            f"Seed {seed}: true label missing for {target}; "
            f"columns={list(df.columns)}"
        )

        assert pred_col is not None, (
            f"Seed {seed}: prediction missing for {target}; "
            f"columns={list(df.columns)}"
        )

        output[f"{target}_true"] = pd.to_numeric(
            df[true_col],
            errors="raise",
        ).astype(int)

        output[f"pred_{target}"] = pd.to_numeric(
            df[pred_col],
            errors="raise",
        ).astype(int)

    assert len(output) == 1050
    assert output["case_uid"].notna().all()
    assert output["case_uid"].nunique() == 1050

    return output.sort_values("case_uid").reset_index(drop=True)


seed_predictions = {
    seed: normalize_prediction_dataframe(path, seed)
    for seed, path in prediction_paths.items()
}

reference_ids = seed_predictions[13]["case_uid"].tolist()

for seed in [42, 100]:
    assert seed_predictions[seed]["case_uid"].tolist() == reference_ids, (
        f"Case ordering or ID mismatch between seed 13 and seed {seed}"
    )

for target in TARGETS:
    reference_true = seed_predictions[13][
        f"{target}_true"
    ].to_numpy()

    for seed in [42, 100]:
        current_true = seed_predictions[seed][
            f"{target}_true"
        ].to_numpy()

        assert np.array_equal(reference_true, current_true), (
            f"Reference-label mismatch for {target}, seed {seed}"
        )

print("Cross-seed ID and label validation: PASSED")


# ------------------------------------------------------------
# 13. Recompute per-seed metrics
# ------------------------------------------------------------

target_metric_rows = []
aggregate_metric_rows = []

for seed in [13, 42, 100]:
    df = seed_predictions[seed]

    macro_values = []

    for target in TARGETS:
        y_true = df[f"{target}_true"].to_numpy()
        y_pred = df[f"pred_{target}"].to_numpy()
        labels = list(range(NUM_LABELS[target]))

        accuracy = accuracy_score(y_true, y_pred)

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_true,
            y_pred,
            labels=labels,
            average="weighted",
            zero_division=0,
        )

        macro_values.append(macro_f1)

        target_metric_rows.append({
            "seed": seed,
            "target": target,
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "weighted_f1": weighted_f1,
            "support": len(df),
        })

    true_matrix = np.column_stack([
        df[f"{target}_true"].to_numpy()
        for target in TARGETS
    ])

    pred_matrix = np.column_stack([
        df[f"pred_{target}"].to_numpy()
        for target in TARGETS
    ])

    joint_em = np.mean(
        np.all(true_matrix == pred_matrix, axis=1)
    )

    aggregate_metric_rows.append({
        "seed": seed,
        "mean_macro_f1": np.mean(macro_values),
        "joint_exact_match": joint_em,
        "support": len(df),
        "seed42_training_mode": (
            "matched_protocol_retraining"
            if seed == 42
            else "matched_protocol"
        ),
    })

target_metrics_df = pd.DataFrame(target_metric_rows)
aggregate_metrics_df = pd.DataFrame(aggregate_metric_rows)


# ------------------------------------------------------------
# 14. Performance variation
# ------------------------------------------------------------

variation_rows = []

for target in TARGETS:
    subset = target_metrics_df.loc[
        target_metrics_df["target"].eq(target)
    ]

    for metric in ["accuracy", "macro_f1", "weighted_f1"]:
        values = subset[metric].to_numpy(dtype=float)

        variation_rows.append({
            "scope": target,
            "metric": metric,
            "n_seeds": len(values),
            "mean": values.mean(),
            "std_sample": values.std(ddof=1),
            "min": values.min(),
            "max": values.max(),
            "range": values.max() - values.min(),
        })

for metric in ["mean_macro_f1", "joint_exact_match"]:
    values = aggregate_metrics_df[metric].to_numpy(dtype=float)

    variation_rows.append({
        "scope": "aggregate",
        "metric": metric,
        "n_seeds": len(values),
        "mean": values.mean(),
        "std_sample": values.std(ddof=1),
        "min": values.min(),
        "max": values.max(),
        "range": values.max() - values.min(),
    })

variation_df = pd.DataFrame(variation_rows)


# ------------------------------------------------------------
# 15. Prediction agreement
# ------------------------------------------------------------

agreement_rows = []
seed_pairs = [(13, 42), (13, 100), (42, 100)]

for target in TARGETS:
    prediction_arrays = {
        seed: seed_predictions[seed][
            f"pred_{target}"
        ].to_numpy()
        for seed in [13, 42, 100]
    }

    for seed_a, seed_b in seed_pairs:
        agreement_rows.append({
            "scope": target,
            "comparison": f"{seed_a}_vs_{seed_b}",
            "agreement_rate": np.mean(
                prediction_arrays[seed_a]
                == prediction_arrays[seed_b]
            ),
            "support": 1050,
        })

    all_agree = (
        (prediction_arrays[13] == prediction_arrays[42])
        & (prediction_arrays[13] == prediction_arrays[100])
    )

    agreement_rows.append({
        "scope": target,
        "comparison": "all_three_seeds",
        "agreement_rate": np.mean(all_agree),
        "support": 1050,
    })

joint_predictions = {}

for seed in [13, 42, 100]:
    joint_predictions[seed] = np.column_stack([
        seed_predictions[seed][
            f"pred_{target}"
        ].to_numpy()
        for target in TARGETS
    ])

for seed_a, seed_b in seed_pairs:
    joint_agreement = np.mean(
        np.all(
            joint_predictions[seed_a]
            == joint_predictions[seed_b],
            axis=1,
        )
    )

    agreement_rows.append({
        "scope": "joint_prediction",
        "comparison": f"{seed_a}_vs_{seed_b}",
        "agreement_rate": joint_agreement,
        "support": 1050,
    })

all_joint_agree = (
    np.all(
        joint_predictions[13] == joint_predictions[42],
        axis=1,
    )
    & np.all(
        joint_predictions[13] == joint_predictions[100],
        axis=1,
    )
)

agreement_rows.append({
    "scope": "joint_prediction",
    "comparison": "all_three_seeds",
    "agreement_rate": np.mean(all_joint_agree),
    "support": 1050,
})

agreement_df = pd.DataFrame(agreement_rows)


# ------------------------------------------------------------
# 16. Save final matched-protocol tables
# ------------------------------------------------------------

target_metrics_path = (
    METRICS_DIR / "per_seed_target_metrics.csv"
)

aggregate_metrics_path = (
    METRICS_DIR / "per_seed_aggregate_metrics.csv"
)

variation_path = (
    METRICS_DIR / "cross_seed_performance_variation.csv"
)

agreement_path = (
    METRICS_DIR / "cross_seed_prediction_agreement.csv"
)

target_metrics_df.to_csv(target_metrics_path, index=False)
aggregate_metrics_df.to_csv(aggregate_metrics_path, index=False)
variation_df.to_csv(variation_path, index=False)
agreement_df.to_csv(agreement_path, index=False)

summary = {
    "experiment_name": EXPERIMENT_NAME,
    "status": "completed_matched_protocol",
    "model_name": MODEL_NAME,
    "dataset": f"{DATASET_NAME}:{DATASET_CONFIG}",
    "seeds": [13, 42, 100],
    "seed_42": {
        "retrained": True,
        "legacy_checkpoint_used": False,
        "best_epoch": best_epoch,
        "checkpoint_path": str(checkpoint_path),
    },
    "per_seed_aggregate_results": (
        aggregate_metrics_df.to_dict(orient="records")
    ),
    "performance_variation": (
        variation_df.to_dict(orient="records")
    ),
    "prediction_agreement": (
        agreement_df.to_dict(orient="records")
    ),
    "privacy": {
        "per_case_predictions_are_private": True,
        "raw_facts_saved": False,
        "checkpoint_publication_recommended": False,
    },
}

summary_path = RESULT_ROOT / "experiment_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)


# ------------------------------------------------------------
# 17. Final console output
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PER-SEED TARGET RESULTS")
print("=" * 72)
print(
    target_metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("PER-SEED AGGREGATE RESULTS")
print("=" * 72)
print(
    aggregate_metrics_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("PERFORMANCE VARIATION")
print("=" * 72)
print(
    variation_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\n" + "=" * 72)
print("PREDICTION AGREEMENT")
print("=" * 72)
print(
    agreement_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

print("\nSaved:")
print(" ", target_metrics_path)
print(" ", aggregate_metrics_path)
print(" ", variation_path)
print(" ", agreement_path)
print(" ", summary_path)
print(" ", seed42_prediction_path)
print(" ", checkpoint_path)

print("\nPRIVACY NOTICE")
print("- Do not upload per-case predictions to GitHub.")
print("- Do not upload model checkpoints unless explicitly intended.")
print("- Aggregate CSV and experiment_summary.json can be reviewed later.")

print("\nMATCHED-PROTOCOL MULTI-SEED REPLICATION: COMPLETED")


## Execution block 2

In [ ]:
import os
# ============================================================
# CONTINUATION CELL
# Generate missing seed-13/100 predictions and aggregate
# DO NOT rerun seed-42 training
# ============================================================

import gc
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import accuracy_score, f1_score


# ------------------------------------------------------------
# 1. Confirm that objects from the previous cell remain loaded
# ------------------------------------------------------------

required_runtime_objects = [
    "PROJECT_ROOT",
    "RESULT_ROOT",
    "CHECKPOINT_ROOT",
    "PREDICTION_DIR",
    "METRICS_DIR",
    "TARGETS",
    "NUM_LABELS",
    "MODEL_NAME",
    "DROPOUT",
    "DEVICE",
    "MultiTaskClassifier",
    "evaluate_model",
    "test_loader",
]

missing_runtime_objects = [
    name
    for name in required_runtime_objects
    if name not in globals()
]

assert not missing_runtime_objects, (
    "The previous training-cell runtime objects are missing: "
    f"{missing_runtime_objects}. "
    "Do not restart the Colab runtime before running this cell."
)

PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 72)
print("MISSING PREDICTION RECOVERY + FINAL AGGREGATION")
print("=" * 72)
print("Device:", DEVICE)
print("Prediction directory:", PREDICTION_DIR)
print("Checkpoint root:", CHECKPOINT_ROOT)


# ------------------------------------------------------------
# 2. Robust checkpoint discovery
# ------------------------------------------------------------

def locate_checkpoint(seed):
    preferred_paths = [
        CHECKPOINT_ROOT / f"seed_{seed}" / "best_model.pt",
        CHECKPOINT_ROOT / f"seed{seed}" / "best_model.pt",
        PROJECT_ROOT
        / "checkpoints"
        / "klue_roberta_base_multiseed"
        / f"seed_{seed}"
        / "best_model.pt",
    ]

    for path in preferred_paths:
        if path.exists():
            return path

    candidates = []

    for path in PROJECT_ROOT.rglob("*.pt"):
        path_string = str(path).lower()
        path_name = path.name.lower()

        seed_patterns = [
            f"seed_{seed}",
            f"seed-{seed}",
            f"seed{seed}",
        ]

        if (
            any(pattern in path_string for pattern in seed_patterns)
            and (
                "best" in path_name
                or "model" in path_name
                or "checkpoint" in path_name
            )
        ):
            candidates.append(path)

    candidates = sorted(set(candidates))

    if not candidates:
        raise FileNotFoundError(
            f"No checkpoint found for seed {seed} under {PROJECT_ROOT}"
        )

    print(f"\nCheckpoint candidates for seed {seed}:")
    for candidate in candidates:
        print(" ", candidate)

    best_model_candidates = [
        path
        for path in candidates
        if path.name == "best_model.pt"
    ]

    if len(best_model_candidates) == 1:
        return best_model_candidates[0]

    matched_protocol_candidates = [
        path
        for path in candidates
        if "klue_roberta_base_multiseed" in str(path)
    ]

    if len(matched_protocol_candidates) == 1:
        return matched_protocol_candidates[0]

    raise RuntimeError(
        f"Multiple ambiguous checkpoints found for seed {seed}. "
        "Select the checkpoint under "
        "checkpoints/klue_roberta_base_multiseed/seed_<seed>/."
    )


checkpoint_paths = {
    seed: locate_checkpoint(seed)
    for seed in [13, 42, 100]
}

print("\nResolved checkpoints:")
for seed, path in checkpoint_paths.items():
    print(f" Seed {seed}: {path}")


# ------------------------------------------------------------
# 3. Extract state dictionary safely
# ------------------------------------------------------------

def extract_state_dict(checkpoint_object):
    if isinstance(checkpoint_object, dict):
        for key in [
            "model_state_dict",
            "state_dict",
            "model",
        ]:
            if key in checkpoint_object:
                candidate = checkpoint_object[key]

                if isinstance(candidate, dict):
                    return candidate

        # 파일 자체가 state_dict인 경우
        if checkpoint_object and all(
            isinstance(key, str)
            for key in checkpoint_object.keys()
        ):
            tensor_value_count = sum(
                torch.is_tensor(value)
                for value in checkpoint_object.values()
            )

            if tensor_value_count > 0:
                return checkpoint_object

    raise RuntimeError(
        "Could not resolve model state_dict from checkpoint."
    )


def remove_module_prefix(state_dict):
    # DataParallel 저장 파일 호환
    if state_dict and all(
        key.startswith("module.")
        for key in state_dict.keys()
    ):
        return {
            key[len("module."):]: value
            for key, value in state_dict.items()
        }

    return state_dict


# ------------------------------------------------------------
# 4. Run test inference for missing seed predictions
# ------------------------------------------------------------

def generate_predictions_from_checkpoint(seed, checkpoint_path):
    print("\n" + "-" * 72)
    print(f"GENERATING TEST PREDICTIONS: SEED {seed}")
    print("-" * 72)
    print("Checkpoint:", checkpoint_path)

    # --------------------------------------------------------
    # 1. Load checkpoint on CPU
    # --------------------------------------------------------

    checkpoint_object = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )

    # 체크포인트 객체에서 실제 model state_dict 추출
    state_dict = extract_state_dict(checkpoint_object)

    # DataParallel로 저장된 경우 module. 접두사 제거
    state_dict = remove_module_prefix(state_dict)

    # --------------------------------------------------------
    # 2. Resolve classifier naming compatibility
    #
    # Older seed-13/100 checkpoints:
    #   classifiers.fine_lv.weight
    #
    # Current model:
    #   heads.fine_lv.weight
    # --------------------------------------------------------

    remapped_state_dict = {}
    remapped_keys = []

    for key, value in state_dict.items():
        if key.startswith("classifiers."):
            new_key = "heads." + key[len("classifiers."):]
            remapped_keys.append((key, new_key))
        else:
            new_key = key

        assert new_key not in remapped_state_dict, (
            "Duplicate state-dict key after remapping: "
            f"{new_key}"
        )

        remapped_state_dict[new_key] = value

    state_dict = remapped_state_dict

    if remapped_keys:
        print(
            "Checkpoint key compatibility mapping:",
            f"{len(remapped_keys)} classifier keys remapped",
        )

        for old_key, new_key in remapped_keys:
            print(f"  {old_key} -> {new_key}")
    else:
        print(
            "Checkpoint key compatibility mapping: "
            "not required"
        )

    # --------------------------------------------------------
    # 3. Initialize current model architecture
    # --------------------------------------------------------

    seed_model = MultiTaskClassifier(
        model_name=MODEL_NAME,
        num_labels=NUM_LABELS,
        dropout=DROPOUT,
    )

    # --------------------------------------------------------
    # 4. Verify tensor shapes before strict loading
    # --------------------------------------------------------

    current_model_state = seed_model.state_dict()

    checkpoint_keys = set(state_dict.keys())
    model_keys = set(current_model_state.keys())

    missing_before_load = sorted(
        model_keys - checkpoint_keys
    )

    unexpected_before_load = sorted(
        checkpoint_keys - model_keys
    )

    shape_mismatches = []

    for key in sorted(model_keys & checkpoint_keys):
        checkpoint_shape = tuple(state_dict[key].shape)
        model_shape = tuple(current_model_state[key].shape)

        if checkpoint_shape != model_shape:
            shape_mismatches.append({
                "key": key,
                "checkpoint_shape": checkpoint_shape,
                "model_shape": model_shape,
            })

    if missing_before_load:
        print("\nMissing keys before loading:")

        for key in missing_before_load:
            print(" ", key)

    if unexpected_before_load:
        print("\nUnexpected keys before loading:")

        for key in unexpected_before_load:
            print(" ", key)

    if shape_mismatches:
        print("\nTensor-shape mismatches:")

        for record in shape_mismatches:
            print(
                f"  {record['key']}: "
                f"checkpoint={record['checkpoint_shape']}, "
                f"model={record['model_shape']}"
            )

    assert not missing_before_load, (
        "Checkpoint is missing model parameters after "
        f"compatibility remapping: {missing_before_load}"
    )

    assert not unexpected_before_load, (
        "Checkpoint contains unexpected parameters after "
        f"compatibility remapping: {unexpected_before_load}"
    )

    assert not shape_mismatches, (
        "Checkpoint contains tensors with incompatible shapes: "
        f"{shape_mismatches}"
    )

    # --------------------------------------------------------
    # 5. Strict checkpoint loading
    # --------------------------------------------------------

    load_result = seed_model.load_state_dict(
        state_dict,
        strict=True,
    )

    print("\nStrict checkpoint loading:")
    print(" Missing keys:", load_result.missing_keys)
    print(" Unexpected keys:", load_result.unexpected_keys)

    assert not load_result.missing_keys
    assert not load_result.unexpected_keys

    seed_model = seed_model.to(DEVICE)

    # --------------------------------------------------------
    # 6. Test-set inference
    # --------------------------------------------------------

    metrics, predictions = evaluate_model(
        seed_model,
        test_loader,
    )

    # --------------------------------------------------------
    # 7. Validate predictions
    # --------------------------------------------------------

    assert len(predictions) == 1050, (
        f"Seed {seed}: expected 1,050 predictions, "
        f"found {len(predictions)}"
    )

    assert predictions["case_uid"].notna().all(), (
        f"Seed {seed}: null case_uid values detected"
    )

    predictions["case_uid"] = (
        predictions["case_uid"].astype(str)
    )

    assert predictions["case_uid"].nunique() == 1050, (
        f"Seed {seed}: duplicate case_uid values detected"
    )

    required_prediction_columns = {
        "case_uid",
        *[
            f"{target}_true"
            for target in TARGETS
        ],
        *[
            f"pred_{target}"
            for target in TARGETS
        ],
    }

    missing_prediction_columns = (
        required_prediction_columns
        - set(predictions.columns)
    )

    assert not missing_prediction_columns, (
        f"Seed {seed}: prediction columns missing: "
        f"{sorted(missing_prediction_columns)}"
    )

    for target in TARGETS:
        true_column = f"{target}_true"
        prediction_column = f"pred_{target}"

        predictions[true_column] = pd.to_numeric(
            predictions[true_column],
            errors="raise",
        ).astype(int)

        predictions[prediction_column] = pd.to_numeric(
            predictions[prediction_column],
            errors="raise",
        ).astype(int)

        valid_values = set(
            range(NUM_LABELS[target])
        )

        true_values = set(
            predictions[true_column].unique()
        )

        predicted_values = set(
            predictions[prediction_column].unique()
        )

        assert true_values.issubset(valid_values), (
            f"Seed {seed}: invalid reference labels "
            f"for {target}: "
            f"{sorted(true_values - valid_values)}"
        )

        assert predicted_values.issubset(valid_values), (
            f"Seed {seed}: invalid predictions "
            f"for {target}: "
            f"{sorted(predicted_values - valid_values)}"
        )

    # --------------------------------------------------------
    # 8. Save private per-case predictions
    # --------------------------------------------------------

    output_path = (
        PREDICTION_DIR
        / f"seed_{seed}_predictions_private.parquet"
    )

    predictions.to_parquet(
        output_path,
        index=False,
    )

    assert output_path.exists(), (
        f"Prediction file was not saved: {output_path}"
    )

    # --------------------------------------------------------
    # 9. Print test performance
    # --------------------------------------------------------

    print(f"\nSeed {seed} test results:")

    for target in TARGETS:
        print(
            f" {target:36s} "
            f"Acc={metrics[target]['accuracy']:.6f} | "
            f"Macro-F1={metrics[target]['macro_f1']:.6f} | "
            f"Weighted-F1="
            f"{metrics[target]['weighted_f1']:.6f}"
        )

    print(
        " Mean Macro-F1:",
        f"{metrics['aggregate']['mean_macro_f1']:.6f}",
    )

    print(
        " Joint Exact Match:",
        f"{metrics['aggregate']['joint_exact_match']:.6f}",
    )

    print(" Cases:", len(predictions))
    print(
        " Unique case_uid:",
        predictions["case_uid"].nunique(),
    )
    print(" Saved:", output_path)

    # --------------------------------------------------------
    # 10. Release memory
    # --------------------------------------------------------

    del seed_model
    del checkpoint_object
    del state_dict
    del remapped_state_dict
    del current_model_state

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return output_path, metrics



# seed 13·100은 체크포인트에서 다시 테스트 추론
# seed 42는 방금 생성한 예측 파일을 그대로 사용
generated_metrics = {}

prediction_paths = {}

for seed in [13, 100]:
    output_path, metrics = generate_predictions_from_checkpoint(
        seed,
        checkpoint_paths[seed],
    )

    prediction_paths[seed] = output_path
    generated_metrics[seed] = metrics

seed42_prediction_path = (
    PREDICTION_DIR
    / "seed_42_predictions_private.parquet"
)

assert seed42_prediction_path.exists(), (
    "The newly trained seed-42 prediction file is missing: "
    f"{seed42_prediction_path}"
)

prediction_paths[42] = seed42_prediction_path

prediction_paths = {
    seed: prediction_paths[seed]
    for seed in [13, 42, 100]
}


# ------------------------------------------------------------
# 5. Load and validate standardized predictions
# ------------------------------------------------------------

def load_standard_prediction(path, seed):
    df = pd.read_parquet(path).copy()

    required_columns = {
        "case_uid",
        *[f"{target}_true" for target in TARGETS],
        *[f"pred_{target}" for target in TARGETS],
    }

    missing_columns = required_columns - set(df.columns)

    assert not missing_columns, (
        f"Seed {seed}: missing columns "
        f"{sorted(missing_columns)}; "
        f"available={list(df.columns)}"
    )

    output_columns = [
        "case_uid",
        *[f"{target}_true" for target in TARGETS],
        *[f"pred_{target}" for target in TARGETS],
    ]

    df = df[output_columns].copy()
    df["case_uid"] = df["case_uid"].astype(str)

    assert len(df) == 1050, (
        f"Seed {seed}: expected 1,050 rows, got {len(df)}"
    )
    assert df["case_uid"].notna().all()
    assert df["case_uid"].nunique() == 1050

    for target in TARGETS:
        true_col = f"{target}_true"
        pred_col = f"pred_{target}"

        df[true_col] = pd.to_numeric(
            df[true_col],
            errors="raise",
        ).astype(int)

        df[pred_col] = pd.to_numeric(
            df[pred_col],
            errors="raise",
        ).astype(int)

        valid_labels = set(range(NUM_LABELS[target]))

        assert set(df[true_col].unique()).issubset(
            valid_labels
        ), f"Seed {seed}: invalid true labels for {target}"

        assert set(df[pred_col].unique()).issubset(
            valid_labels
        ), f"Seed {seed}: invalid predictions for {target}"

    return df.sort_values("case_uid").reset_index(drop=True)


seed_predictions = {
    seed: load_standard_prediction(path, seed)
    for seed, path in prediction_paths.items()
}

reference_ids = seed_predictions[13]["case_uid"].to_numpy()

for seed in [42, 100]:
    current_ids = seed_predictions[seed][
        "case_uid"
    ].to_numpy()

    assert np.array_equal(reference_ids, current_ids), (
        f"Case-ID mismatch between seed 13 and seed {seed}"
    )

for target in TARGETS:
    reference_labels = seed_predictions[13][
        f"{target}_true"
    ].to_numpy()

    for seed in [42, 100]:
        current_labels = seed_predictions[seed][
            f"{target}_true"
        ].to_numpy()

        assert np.array_equal(
            reference_labels,
            current_labels,
        ), (
            f"Reference-label mismatch: "
            f"target={target}, seed={seed}"
        )

print("\nCross-seed validation:")
print(" Cases: 1050")
print(" Unique case IDs: 1050")
print(" Case-ID alignment: PASSED")
print(" Reference-label alignment: PASSED")


# ------------------------------------------------------------
# 6. Recompute per-seed performance
# ------------------------------------------------------------

target_metric_rows = []
aggregate_metric_rows = []

for seed in [13, 42, 100]:
    df = seed_predictions[seed]

    seed_macro_f1_values = []

    for target in TARGETS:
        y_true = df[f"{target}_true"].to_numpy()
        y_pred = df[f"pred_{target}"].to_numpy()

        target_labels = list(
            range(NUM_LABELS[target])
        )

        accuracy = accuracy_score(y_true, y_pred)

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=target_labels,
            average="macro",
            zero_division=0,
        )

        weighted_f1 = f1_score(
            y_true,
            y_pred,
            labels=target_labels,
            average="weighted",
            zero_division=0,
        )

        seed_macro_f1_values.append(macro_f1)

        target_metric_rows.append({
            "seed": seed,
            "target": target,
            "accuracy": float(accuracy),
            "macro_f1": float(macro_f1),
            "weighted_f1": float(weighted_f1),
            "support": len(df),
        })

    true_matrix = np.column_stack([
        df[f"{target}_true"].to_numpy()
        for target in TARGETS
    ])

    pred_matrix = np.column_stack([
        df[f"pred_{target}"].to_numpy()
        for target in TARGETS
    ])

    joint_exact_match = float(
        np.mean(
            np.all(
                true_matrix == pred_matrix,
                axis=1,
            )
        )
    )

    aggregate_metric_rows.append({
        "seed": seed,
        "mean_macro_f1": float(
            np.mean(seed_macro_f1_values)
        ),
        "joint_exact_match": joint_exact_match,
        "support": len(df),
        "protocol": "matched_protocol",
    })

target_metrics_df = pd.DataFrame(target_metric_rows)
aggregate_metrics_df = pd.DataFrame(
    aggregate_metric_rows
)


# ------------------------------------------------------------
# 7. Cross-seed performance variation
# ------------------------------------------------------------

variation_rows = []

for target in TARGETS:
    target_subset = target_metrics_df.loc[
        target_metrics_df["target"].eq(target)
    ]

    for metric in [
        "accuracy",
        "macro_f1",
        "weighted_f1",
    ]:
        values = target_subset[metric].to_numpy(
            dtype=float
        )

        variation_rows.append({
            "scope": target,
            "metric": metric,
            "n_seeds": len(values),
            "mean": float(values.mean()),
            "std_sample": float(values.std(ddof=1)),
            "min": float(values.min()),
            "max": float(values.max()),
            "range": float(
                values.max() - values.min()
            ),
        })

for metric in [
    "mean_macro_f1",
    "joint_exact_match",
]:
    values = aggregate_metrics_df[metric].to_numpy(
        dtype=float
    )

    variation_rows.append({
        "scope": "aggregate",
        "metric": metric,
        "n_seeds": len(values),
        "mean": float(values.mean()),
        "std_sample": float(values.std(ddof=1)),
        "min": float(values.min()),
        "max": float(values.max()),
        "range": float(
            values.max() - values.min()
        ),
    })

variation_df = pd.DataFrame(variation_rows)


# ------------------------------------------------------------
# 8. Cross-seed prediction agreement
# ------------------------------------------------------------

agreement_rows = []
seed_pairs = [
    (13, 42),
    (13, 100),
    (42, 100),
]

for target in TARGETS:
    prediction_arrays = {
        seed: seed_predictions[seed][
            f"pred_{target}"
        ].to_numpy()
        for seed in [13, 42, 100]
    }

    for seed_a, seed_b in seed_pairs:
        agreement_rate = float(
            np.mean(
                prediction_arrays[seed_a]
                == prediction_arrays[seed_b]
            )
        )

        agreement_rows.append({
            "scope": target,
            "comparison": f"{seed_a}_vs_{seed_b}",
            "agreement_rate": agreement_rate,
            "disagreement_rate": 1.0 - agreement_rate,
            "support": 1050,
        })

    all_three_agree = (
        (prediction_arrays[13]
         == prediction_arrays[42])
        &
        (prediction_arrays[13]
         == prediction_arrays[100])
    )

    all_three_rate = float(
        np.mean(all_three_agree)
    )

    agreement_rows.append({
        "scope": target,
        "comparison": "all_three_seeds",
        "agreement_rate": all_three_rate,
        "disagreement_rate": 1.0 - all_three_rate,
        "support": 1050,
    })

joint_predictions = {
    seed: np.column_stack([
        seed_predictions[seed][
            f"pred_{target}"
        ].to_numpy()
        for target in TARGETS
    ])
    for seed in [13, 42, 100]
}

for seed_a, seed_b in seed_pairs:
    agreement_rate = float(
        np.mean(
            np.all(
                joint_predictions[seed_a]
                == joint_predictions[seed_b],
                axis=1,
            )
        )
    )

    agreement_rows.append({
        "scope": "joint_prediction",
        "comparison": f"{seed_a}_vs_{seed_b}",
        "agreement_rate": agreement_rate,
        "disagreement_rate": 1.0 - agreement_rate,
        "support": 1050,
    })

all_joint_agree = (
    np.all(
        joint_predictions[13]
        == joint_predictions[42],
        axis=1,
    )
    &
    np.all(
        joint_predictions[13]
        == joint_predictions[100],
        axis=1,
    )
)

all_joint_agreement_rate = float(
    np.mean(all_joint_agree)
)

agreement_rows.append({
    "scope": "joint_prediction",
    "comparison": "all_three_seeds",
    "agreement_rate": all_joint_agreement_rate,
    "disagreement_rate": (
        1.0 - all_joint_agreement_rate
    ),
    "support": 1050,
})

agreement_df = pd.DataFrame(agreement_rows)


# ------------------------------------------------------------
# 9. Save final matched-protocol outputs
# ------------------------------------------------------------

target_metrics_path = (
    METRICS_DIR
    / "per_seed_target_metrics.csv"
)

aggregate_metrics_path = (
    METRICS_DIR
    / "per_seed_aggregate_metrics.csv"
)

variation_path = (
    METRICS_DIR
    / "cross_seed_performance_variation.csv"
)

agreement_path = (
    METRICS_DIR
    / "cross_seed_prediction_agreement.csv"
)

target_metrics_df.to_csv(
    target_metrics_path,
    index=False,
)

aggregate_metrics_df.to_csv(
    aggregate_metrics_path,
    index=False,
)

variation_df.to_csv(
    variation_path,
    index=False,
)

agreement_df.to_csv(
    agreement_path,
    index=False,
)

summary = {
    "experiment_name": (
        "baseline_multiseed_replication_v1"
    ),
    "status": "completed_matched_protocol",
    "model_name": MODEL_NAME,
    "seeds": [13, 42, 100],
    "seed_42_legacy_checkpoint_used": False,
    "seed_42_retrained_under_matched_protocol": True,
    "test_cases": 1050,
    "per_seed_aggregate_results": (
        aggregate_metrics_df.to_dict(
            orient="records"
        )
    ),
    "performance_variation": (
        variation_df.to_dict(
            orient="records"
        )
    ),
    "prediction_agreement": (
        agreement_df.to_dict(
            orient="records"
        )
    ),
    "privacy": {
        "per_case_predictions_private": True,
        "raw_facts_saved": False,
        "aggregate_metrics_publishable": True,
    },
}

summary_path = (
    RESULT_ROOT
    / "experiment_summary.json"
)

with open(
    summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ------------------------------------------------------------
# 10. Print requested final results
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("PER-SEED TARGET RESULTS")
print("=" * 72)
print(
    target_metrics_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("PER-SEED AGGREGATE RESULTS")
print("=" * 72)
print(
    aggregate_metrics_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("PERFORMANCE VARIATION")
print("=" * 72)
print(
    variation_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\n" + "=" * 72)
print("PREDICTION AGREEMENT")
print("=" * 72)
print(
    agreement_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nSaved outputs:")
print(" ", target_metrics_path)
print(" ", aggregate_metrics_path)
print(" ", variation_path)
print(" ", agreement_path)
print(" ", summary_path)

print("\nPrivate prediction files:")
for seed, path in prediction_paths.items():
    print(f" Seed {seed}: {path}")

print("\nFINAL MATCHED-PROTOCOL MULTI-SEED AGGREGATION: COMPLETED")
